# Significances

Plot a scan of significances.

In [1]:
import os
import glob
import uproot
import matplotlib.pyplot as plt
import plot_utils
from prettytable import PrettyTable

In [2]:
motherDir = "../Combined_cards/Combined/WH/"
subDirs = glob.glob(motherDir + "cards-*")

In [4]:
# filling things based on structure of /Combined_cards/Combined/, and the result of running the significances_zh.sh and significances_wh.sh scripts in ZH and WH subdirectories accordingly

significances_WH = []
for subDir in subDirs:
    _f = uproot.open(subDir + "/higgsCombineTest.Significance.mH120.root")
    _significance = _f['limit']['limit'].array()
    _sample_params = list(plot_utils.get_params_from_sample_name_wh(subDir, save_mS=False))
    _sample_params[-1] = plot_utils.mA[_sample_params[-1]]
    significances_WH.append((_sample_params, _significance[0]))

significances_ZH = []
files = glob.glob("../Combined_cards/Combined/ZH/*Significance.mH120.root")
for file in files:
    _f = uproot.open(file)
    _significance = _f['limit']['limit'].array()
    _sample_params = list(plot_utils.get_params_from_sample_name_zh_no_mS(file))
    _sample_params[-1] = plot_utils.mA[_sample_params[-1]]
    significances_ZH.append((_sample_params, _significance[0]))

significances_VH = []
files = glob.glob("../Combined_cards/Combined/*Significance.mH120.root")
for file in files:
    _f = uproot.open(file)
    _significance = _f['limit']['limit'].array()
    _sample_params = list(plot_utils.get_params_from_sample_name_vh(file))
    _sample_params[-1] = plot_utils.mA[_sample_params[-1]]
    significances_VH.append((_sample_params, _significance[0]))

In [5]:
# Assume these lists exist and have been filled:
# significances_WH = [ ([mD, T, mA], significance), ... ]
# significances_ZH = [ ([mD, T, mA], significance), ... ]
# significances_VH = [ ([mD, T, mA], significance), ... ]

# Build dictionaries for fast lookup, using the parameter list (converted to a tuple) as the key.
dict_WH = {tuple(params): signif for params, signif in significances_WH}
dict_ZH = {tuple(params): signif for params, signif in significances_ZH}
dict_VH = {tuple(params): signif for params, signif in significances_VH}

# only keep points that exist for both WH and ZH !
common_keys = set(dict_WH.keys()) & set(dict_ZH.keys()) & set(dict_VH.keys())

sorted_keys = sorted(common_keys, key=lambda key: (key[0], key[1], key[2]))

table_rows = []
for key in sorted_keys:
    WHsig = dict_WH[key]
    ZHsig = dict_ZH[key]
    VHsig = dict_VH[key]
    
    # Only keep the sample if VH significance exceeds both WH and ZH significances. !!!!!!!
    if VHsig > WHsig and VHsig > ZHsig:
        # ignore tiny fluctuations around 0 significance
        if WHsig < 0.1 and ZHsig < 0.1 and VHsig < 0.1:
            continue
        mD, T, mA = key
        table_rows.append([mD, T, mA, WHsig, ZHsig, VHsig])

table = PrettyTable()
table.field_names = ["mD [GeV]", "T [GeV]", "mA [GeV]", "WH significance", "ZH significance", "VH significance"]

for row in table_rows:
    row_formatted = row[:3] + [round(val, 3) for val in row[3:]]
    table.add_row(row_formatted)

print(table)


+----------+---------+----------+-----------------+-----------------+-----------------+
| mD [GeV] | T [GeV] | mA [GeV] | WH significance | ZH significance | VH significance |
+----------+---------+----------+-----------------+-----------------+-----------------+
|   8.0    |   32.0  |   0.7    |      0.031      |      0.209      |      0.323      |
+----------+---------+----------+-----------------+-----------------+-----------------+


In [6]:
########### same as above but for all the points......

dict_WH = {tuple(params): signif for params, signif in significances_WH}
dict_ZH = {tuple(params): signif for params, signif in significances_ZH}
dict_VH = {tuple(params): signif for params, signif in significances_VH}

common_keys = set(dict_WH.keys()) & set(dict_ZH.keys()) & set(dict_VH.keys())

sorted_keys = sorted(common_keys, key=lambda key: (key[0], key[1], key[2]))

table_rows = []
for key in sorted_keys:
    mD, T, mA = key
    row = [mD, T, mA, dict_WH[key], dict_ZH[key], dict_VH[key]]
    table_rows.append(row)

table = PrettyTable()
table.field_names = ["mD [GeV]", "T [GeV]", "mA [GeV]", "WH significance", "ZH significance", "VH significance"]

for row in table_rows:
    row_formatted = row[:3] + [round(val, 3) for val in row[3:]]
    table.add_row(row_formatted)

print(table)


+----------+---------+----------+-----------------+-----------------+-----------------+
| mD [GeV] | T [GeV] | mA [GeV] | WH significance | ZH significance | VH significance |
+----------+---------+----------+-----------------+-----------------+-----------------+
|   1.0    |   0.25  |   0.5    |       0.0       |       0.0       |       0.0       |
|   1.0    |   0.5   |   0.5    |       0.0       |       0.0       |       0.0       |
|   1.0    |   1.0   |   0.5    |       0.0       |       0.0       |       0.0       |
|   1.0    |   2.0   |   0.5    |      0.001      |       0.0       |      0.001      |
|   1.0    |   4.0   |   0.5    |       0.0       |       0.0       |       0.0       |
|   1.4    |   0.35  |   0.7    |       0.0       |       0.0       |       0.0       |
|   1.4    |   0.7   |   0.7    |       0.0       |       0.0       |       0.0       |
|   1.4    |   1.4   |   0.7    |       0.0       |       0.0       |       0.0       |
|   1.4    |   2.8   |   0.7    

In [7]:
keys_WH = set(tuple(item[0]) for item in significances_WH)
keys_ZH = set(tuple(item[0]) for item in significances_ZH)
keys_VH = set(tuple(item[0]) for item in significances_VH)

common_keys = keys_WH & keys_ZH & keys_VH

print("Common sample parameters:", common_keys)

significances_WH_common = [item for item in significances_WH if tuple(item[0]) in common_keys]
significances_ZH_common = [item for item in significances_ZH if tuple(item[0]) in common_keys]
significances_VH_common = [item for item in significances_VH if tuple(item[0]) in common_keys]

Common sample parameters: {(4.0, 4.0, 0.7), (8.0, 8.0, 0.5), (2.0, 1.0, 0.5), (2.0, 0.5, 1.0), (4.0, 4.0, 1.0), (3.0, 1.5, 0.7), (4.0, 16.0, 0.5), (4.0, 2.0, 1.0), (2.0, 0.5, 0.7), (3.0, 6.0, 0.7), (8.0, 16.0, 0.7), (3.0, 1.5, 1.0), (3.0, 6.0, 1.0), (8.0, 32.0, 0.5), (4.0, 8.0, 0.7), (3.0, 0.75, 0.5), (8.0, 16.0, 1.0), (1.0, 1.0, 0.5), (2.0, 2.0, 0.7), (4.0, 8.0, 1.0), (1.0, 0.5, 0.5), (1.4, 0.7, 0.7), (2.0, 0.5, 0.5), (3.0, 12.0, 0.7), (4.0, 2.0, 0.5), (2.0, 2.0, 1.0), (3.0, 3.0, 0.7), (4.0, 1.0, 0.7), (2.0, 4.0, 0.7), (4.0, 4.0, 0.5), (3.0, 12.0, 1.0), (8.0, 2.0, 0.7), (3.0, 3.0, 1.0), (8.0, 4.0, 0.7), (3.0, 1.5, 0.5), (2.0, 8.0, 0.7), (8.0, 2.0, 1.0), (3.0, 6.0, 0.5), (4.0, 1.0, 1.0), (2.0, 4.0, 1.0), (8.0, 4.0, 1.0), (8.0, 8.0, 0.7), (8.0, 16.0, 0.5), (2.0, 1.0, 0.7), (1.4, 5.6, 0.7), (1.4, 0.35, 0.7), (4.0, 16.0, 0.7), (2.0, 8.0, 1.0), (4.0, 8.0, 0.5), (8.0, 8.0, 1.0), (2.0, 2.0, 0.5), (2.0, 1.0, 1.0), (1.4, 2.8, 0.7), (8.0, 32.0, 0.7), (1.0, 2.0, 0.5), (3.0, 0.75, 0.7), (4.0, 16.